# 02 Exploratory Analysis

This notebook presents exploratory data analysis for the cleaned and merged spare parts dataset.

The notebook keeps the existing workflow simple and separates the analysis into:
1. historical dataset analysis
2. latest snapshot dataset analysis

The purpose is descriptive. No modelling is introduced here.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid")


## Load dataset

The cleaned and merged dataset is loaded first. Each row is a scraped listing observation.


In [ ]:
df = pd.read_csv("../../datasets/cleaned/cleaned_and_merged_pricedataset.csv")
df["scrape_date"] = pd.to_datetime(df["scrape_date"])

df.head()


## What `df` and `latest_df` represent

- `df` contains the full historical dataset across all scrape dates.
- `latest_df` will later contain only the latest available observation for each `product_id`.

This distinction matters because the same product can appear multiple times in the historical data.


## Dataset overview

This section gives a compact summary of dataset size and time coverage.


In [ ]:
dataset_summary = pd.DataFrame()
dataset_summary["rows"] = [len(df)]
dataset_summary["columns"] = [len(df.columns)]
dataset_summary["unique_products"] = [df["product_id"].nunique()]
dataset_summary["unique_scrape_dates"] = [df["scrape_date"].nunique()]
dataset_summary["start_date"] = [df["scrape_date"].min().date()]
dataset_summary["end_date"] = [df["scrape_date"].max().date()]

dataset_summary


The dataset covers six scrape dates in February 2026 and contains repeated product observations over time.


## Data quality and duplicate diagnostics

These checks focus especially on redundant duplicates involving `product_id` and `scrape_date`.


In [ ]:
quality_table = pd.DataFrame()
quality_table["exact_duplicate_rows"] = [df.duplicated().sum()]
quality_table["duplicate_product_id_rows"] = [df.duplicated(subset=["product_id"]).sum()]
quality_table["duplicate_product_id_scrape_date_rows"] = [df.duplicated(subset=["product_id", "scrape_date"]).sum()]
quality_table["missing_price"] = [df["price"].isna().sum()]
quality_table["missing_mileage"] = [df["mileage"].isna().sum()]
quality_table["missing_quality_grade"] = [df["quality_grade"].isna().sum()]

quality_table


In [ ]:
duplicate_product_date_rows = df[df.duplicated(subset=["product_id", "scrape_date"], keep=False)]
duplicate_product_date_rows = duplicate_product_date_rows.sort_values(["product_id", "scrape_date"])

print("Rows with duplicate product_id + scrape_date:", len(duplicate_product_date_rows))

duplicate_product_date_rows.head(20)


If `product_id + scrape_date` duplicates exist, they may indicate redundant same-day records rather than normal longitudinal tracking across different dates.


## Missing values

Missingness is examined overall and then specifically for `mileage`.


In [ ]:
missing_table = pd.DataFrame()
missing_table["missing_count"] = df.isna().sum()
missing_table["missing_percent"] = (df.isna().sum() / len(df)) * 100
missing_table = missing_table.sort_values("missing_count", ascending=False)

missing_table


In [ ]:
mileage_missing_by_category = df.groupby("category").agg(
    total_rows=("mileage", "size"),
    missing_mileage=("mileage", lambda x: x.isna().sum())
)

mileage_missing_by_category["missing_mileage_percent"] = (
    mileage_missing_by_category["missing_mileage"] / mileage_missing_by_category["total_rows"]
) * 100

mileage_missing_by_category = mileage_missing_by_category.reset_index()
mileage_missing_by_category = mileage_missing_by_category.sort_values("missing_mileage_percent", ascending=False)

mileage_missing_by_category


Mileage is the main incomplete variable, and the missingness can vary by category.


## Historical dataset analysis

This part uses the full dataset `df` and therefore includes repeated observations of the same product across scrape dates.


In [ ]:
historical_summary = df.groupby("scrape_date").agg(
    rows=("product_id", "size"),
    unique_products=("product_id", "nunique"),
    mean_price=("price", "mean"),
    median_price=("price", "median"),
    median_mileage=("mileage", "median")
)

historical_summary = historical_summary.reset_index()

historical_summary


The historical summary shows how listing counts and central price levels change across the scrape dates.


In [ ]:
daily_brand_counts = df.groupby(["scrape_date", "brand"])["product_id"].nunique().reset_index()
daily_brand_counts = daily_brand_counts.rename(columns={"product_id": "unique_products"})

plt.figure(figsize=(12, 6))
sns.lineplot(data=daily_brand_counts, x="scrape_date", y="unique_products", hue="brand", marker="o", linewidth=2.2)
plt.title("Unique observed products over time by brand")
plt.xlabel("Scrape date")
plt.ylabel("Unique products")
plt.xticks(rotation=45)
plt.legend(title="Brand")
plt.tight_layout()
plt.show()


This figure describes how the observed market size changes over time for each brand.


In [ ]:
daily_brand_price = df.groupby(["scrape_date", "brand"])["price"].median().reset_index()
daily_brand_price = daily_brand_price.rename(columns={"price": "median_price"})

plt.figure(figsize=(12, 6))
sns.lineplot(data=daily_brand_price, x="scrape_date", y="median_price", hue="brand", marker="o", linewidth=2.2)
plt.title("Median listing price over time by brand")
plt.xlabel("Scrape date")
plt.ylabel("Median price")
plt.xticks(rotation=45)
plt.legend(title="Brand")
plt.tight_layout()
plt.show()


Median prices are shown here because they are less affected by high-value outliers than mean prices.


## Latest snapshot dataset analysis

This part uses one final observation per `product_id`, which is more suitable for cross-sectional comparisons.


In [ ]:
latest_df = df.sort_values(["product_id", "scrape_date"])
latest_df = latest_df.drop_duplicates(subset=["product_id"], keep="last")
latest_df = latest_df.reset_index(drop=True)

latest_summary = pd.DataFrame()
latest_summary["rows"] = [len(latest_df)]
latest_summary["unique_products"] = [latest_df["product_id"].nunique()]
latest_summary["brands"] = [latest_df["brand"].nunique()]
latest_summary["categories"] = [latest_df["category"].nunique()]

latest_summary


The latest snapshot removes repeated time observations and keeps one row per product listing.


In [ ]:
snapshot_overview = latest_df.groupby(["brand", "model"]).agg(
    listings=("product_id", "count"),
    median_price=("price", "median"),
    mean_price=("price", "mean"),
    median_mileage=("mileage", "median"),
    median_year_mid=("year_mid", "median")
)

snapshot_overview = snapshot_overview.reset_index()
snapshot_overview = snapshot_overview.sort_values("listings", ascending=False)

snapshot_overview


This table compares the main brand-model groups in the latest available market snapshot.


In [ ]:
brand_table = latest_df.groupby("brand").agg(
    listings=("product_id", "count"),
    median_price=("price", "median"),
    mean_price=("price", "mean"),
    median_mileage=("mileage", "median")
)

brand_table = brand_table.reset_index()
brand_table = brand_table.sort_values("median_price", ascending=False)

brand_table


In [ ]:
category_table = latest_df.groupby("category").agg(
    listings=("product_id", "count"),
    median_price=("price", "median"),
    mean_price=("price", "mean"),
    median_mileage=("mileage", "median")
)

category_table = category_table.reset_index()
category_table = category_table.sort_values(["listings", "median_price"], ascending=[False, False])

category_table


Category-level summaries help identify where listing volume and price levels are concentrated.


## Price analysis by quality grade

This section checks whether price levels vary across quality grades.


In [ ]:
quality_grade_table = latest_df.groupby("quality_grade").agg(
    listings=("product_id", "count"),
    median_price=("price", "median"),
    mean_price=("price", "mean"),
    median_mileage=("mileage", "median")
)

quality_grade_table = quality_grade_table.reset_index()
quality_grade_table = quality_grade_table.sort_values("median_price", ascending=False)

quality_grade_table


In [ ]:
quality_plot_df = latest_df.dropna(subset=["quality_grade"]).copy()

plt.figure(figsize=(9, 6))
sns.boxplot(data=quality_plot_df, x="quality_grade", y="price")
plt.title("Price distribution by quality grade")
plt.xlabel("Quality grade")
plt.ylabel("Price")
plt.tight_layout()
plt.show()


Differences across quality grades should be interpreted descriptively because price also depends on part type, brand and other characteristics.


## Category counts

The next figure shows the largest categories in the latest snapshot.


In [ ]:
top_category_counts = latest_df["category"].value_counts().head(10)
top_category_counts = top_category_counts.reset_index()
top_category_counts.columns = ["category", "count"]

plt.figure(figsize=(12, 6))
sns.barplot(data=top_category_counts, x="count", y="category", palette="Blues_r")
plt.title("Largest categories by listing count")
plt.xlabel("Count")
plt.ylabel("Category")
plt.tight_layout()
plt.show()


This plot identifies which categories account for the largest share of current listings.


## Price distributions

Because the price variable is right-skewed, both the raw and log-transformed distributions are shown.


In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(latest_df["price"], bins=40, kde=True, color="steelblue")
plt.title("Distribution of price")
plt.xlabel("Price")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
log_price = np.log1p(latest_df["price"])

plt.figure(figsize=(10, 6))
sns.histplot(log_price, bins=40, kde=True, color="darkorange")
plt.title("Distribution of log1p(price)")
plt.xlabel("log1p(price)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


The log-transformed price distribution provides a clearer view of the central mass of the data.


In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(latest_df["mileage"].dropna(), bins=40, kde=True, color="seagreen")
plt.title("Distribution of mileage")
plt.xlabel("Mileage")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## Price by category

The standard and zoomed boxplots below show the same categories at different scales.


In [ ]:
top_categories = latest_df["category"].value_counts().head(6).index
plot_df = latest_df[latest_df["category"].isin(top_categories)].copy()

plt.figure(figsize=(12, 7))
sns.boxplot(data=plot_df, x="category", y="price")
plt.title("Price distribution across the largest categories")
plt.xlabel("Category")
plt.ylabel("Price")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
price_limit = latest_df["price"].quantile(0.95)

plt.figure(figsize=(12, 7))
sns.boxplot(data=plot_df, x="category", y="price")
plt.ylim(0, price_limit)
plt.title("Price by category (zoomed to the 95th percentile)")
plt.xlabel("Category")
plt.ylabel("Price")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


The zoomed version makes typical category-level differences easier to read because the upper tail no longer dominates the scale.


## Price and mileage relationship

These plots remain descriptive and should not be treated as causal evidence.


In [ ]:
plt.figure(figsize=(12, 7))
sns.scatterplot(data=latest_df, x="mileage", y="price", hue="brand", alpha=0.65)
plt.title("Mileage versus price in the latest snapshot")
plt.xlabel("Mileage")
plt.ylabel("Price")
plt.legend(title="Brand")
plt.tight_layout()
plt.show()


In [ ]:
regression_df = latest_df[["mileage", "price"]].dropna()

plt.figure(figsize=(12, 7))
sns.regplot(data=regression_df, x="mileage", y="price", scatter_kws={"alpha": 0.25}, line_kws={"color": "red"})
plt.title("Relationship between mileage and price")
plt.xlabel("Mileage")
plt.ylabel("Price")
plt.tight_layout()
plt.show()


## Correlation matrix

The correlation matrix is presented as a compact descriptive summary. It should not be overinterpreted.


In [ ]:
corr_columns = ["price", "mileage", "year_start", "year_end", "year_span", "year_mid"]
corr = latest_df[corr_columns].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, cmap="crest", fmt=".2f", square=True)
plt.title("Correlation matrix for numerical variables")
plt.tight_layout()
plt.show()


Correlations among the year variables are expected because they describe closely related model-year information. The more useful descriptive relationships are those involving price and mileage.


## Key findings

- The historical dataset contains repeated product observations across scrape dates, which is expected in longitudinal scraping.
- The latest snapshot dataset is more appropriate for comparing current price levels across brands, categories and quality grades.
- The dataset contains no exact full-row duplicates, but duplicate checks using `product_id` and `product_id + scrape_date` remain important for identifying redundant records.
- Price is right-skewed, so median-based summaries and the `log1p(price)` histogram are useful descriptive tools.
- Mileage is the main incomplete variable, and its missingness differs across categories.
- Category and quality-grade comparisons suggest meaningful variation in price levels, but these differences should be interpreted descriptively rather than causally.
